# IMDb 2024 Movie Scraping

This notebook scrapes IMDb 2024 feature films using Selenium and collects two columns:
- `Movie_Title`
- `Storyline`

The target is up to 6000 movies. The final dataset is saved as `imdb_movies_2024_6000.csv`.

## 1. Import Libraries

In [1]:
import time
import pandas as pd

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

print('Libraries imported successfully.')

Libraries imported successfully.


## 2. Set Target and IMDb URL

This URL filters IMDb for feature films released from January 1, 2024 to December 31, 2024.

In [2]:
TARGET_MOVIES = 6000

URL = (
    'https://www.imdb.com/search/title/'
    '?title_type=feature'
    '&release_date=2024-01-01,2024-12-31'
    '&count=250'
)

print(URL)

https://www.imdb.com/search/title/?title_type=feature&release_date=2024-01-01,2024-12-31&count=250


## 3. Open IMDb in Chrome

If IMDb shows a CAPTCHA, complete it manually in the Chrome window before continuing.

In [3]:
driver = webdriver.Chrome()
driver.maximize_window()
driver.get(URL)

print('IMDb opened in Chrome.')
input('Complete CAPTCHA if it appears, then press Enter here...')

IMDb opened in Chrome.


''

## 4. Wait for the Initial Movie List

In [4]:
wait = WebDriverWait(driver, 20)

wait.until(
    EC.presence_of_all_elements_located(
        (By.CSS_SELECTOR, 'li.ipc-metadata-list-summary-item')
    )
)

print('Initial movie list loaded successfully.')

Initial movie list loaded successfully.


## 5. Load More Movies Until the Target Is Reached

The notebook repeatedly scrolls to the bottom and clicks IMDb's `250 more` button. It stops when 6000 movie cards are loaded or when no more results are available.

In [3]:
import time
import pandas as pd

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

print("Libraries imported successfully.")

Libraries imported successfully.


In [4]:
TARGET_MOVIES = 6000

URL = (
    "https://www.imdb.com/search/title/"
    "?title_type=feature"
    "&release_date=2024-01-01,2024-12-31"
    "&count=250"
)

driver = webdriver.Chrome()
driver.maximize_window()

driver.get(URL)

print("IMDb opened successfully.")

IMDb opened successfully.


In [5]:
wait = WebDriverWait(driver, 30)

wait.until(
    EC.presence_of_all_elements_located(
        (
            By.CSS_SELECTOR,
            "li.ipc-metadata-list-summary-item"
        )
    )
)

movie_elements = driver.find_elements(
    By.CSS_SELECTOR,
    "li.ipc-metadata-list-summary-item"
)

print("Movies currently available:", len(movie_elements))

Movies currently available: 250


In [6]:
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time

TARGET_MOVIES = 6000

while True:
    movie_elements = driver.find_elements(
        By.CSS_SELECTOR,
        "li.ipc-metadata-list-summary-item"
    )

    current_count = len(movie_elements)
    print(f"Movies currently loaded: {current_count}")

    if current_count >= TARGET_MOVIES:
        print(f"Target of {TARGET_MOVIES} movies reached.")
        break

    # Scroll to bottom so the Load More button becomes visible
    driver.execute_script(
        "window.scrollTo(0, document.body.scrollHeight);"
    )

    time.sleep(4)

    try:
        # Find the actual button containing "250 more"
        load_more_button = WebDriverWait(driver, 15).until(
            EC.presence_of_element_located(
                (
                    By.XPATH,
                    "//button[.//span[contains(normalize-space(), '250 more')]]"
                )
            )
        )

        driver.execute_script(
            "arguments[0].scrollIntoView({block:'center'});",
            load_more_button
        )

        time.sleep(2)

        previous_count = current_count

        # Click using JavaScript
        driver.execute_script(
            "arguments[0].click();",
            load_more_button
        )

        print("Clicked '250 more'.")

        # Wait until the movie count actually increases
        try:
            WebDriverWait(driver, 30).until(
                lambda d: len(
                    d.find_elements(
                        By.CSS_SELECTOR,
                        "li.ipc-metadata-list-summary-item"
                    )
                ) > previous_count
            )

            new_count = len(
                driver.find_elements(
                    By.CSS_SELECTOR,
                    "li.ipc-metadata-list-summary-item"
                )
            )

            print(f"New movie count: {new_count}")

        except Exception:
            print("Count did not increase after clicking.")
            print("Trying again after extra scrolling...")

            # Extra scrolling can trigger lazy loading
            for _ in range(5):
                driver.execute_script(
                    "window.scrollBy(0, 1000);"
                )
                time.sleep(1)

            new_count = len(
                driver.find_elements(
                    By.CSS_SELECTOR,
                    "li.ipc-metadata-list-summary-item"
                )
            )

            print(f"Count after retry: {new_count}")

            if new_count <= previous_count:
                print("No new movies were loaded. Stopping.")
                break

        time.sleep(3)

    except Exception as e:
        print("Load More button could not be found.")
        print("Reason:", e)
        break

Movies currently loaded: 250
Clicked '250 more'.
New movie count: 500
Movies currently loaded: 500
Clicked '250 more'.
New movie count: 750
Movies currently loaded: 750
Clicked '250 more'.
New movie count: 1000
Movies currently loaded: 1000
Clicked '250 more'.
New movie count: 1250
Movies currently loaded: 1250
Clicked '250 more'.
New movie count: 1500
Movies currently loaded: 1500
Clicked '250 more'.
New movie count: 1750
Movies currently loaded: 1750
Clicked '250 more'.
New movie count: 2000
Movies currently loaded: 2000
Clicked '250 more'.
New movie count: 2250
Movies currently loaded: 2250
Clicked '250 more'.
New movie count: 2500
Movies currently loaded: 2500
Clicked '250 more'.
New movie count: 2750
Movies currently loaded: 2750
Clicked '250 more'.
New movie count: 3000
Movies currently loaded: 3000
Clicked '250 more'.
New movie count: 3250
Movies currently loaded: 3250
Clicked '250 more'.
New movie count: 3500
Movies currently loaded: 3500
Clicked '250 more'.
New movie count: 37

## 6. Select the First 6000 Movie Cards

In [7]:
movie_elements = driver.find_elements(
    By.CSS_SELECTOR,
    'li.ipc-metadata-list-summary-item'
)

movie_elements = movie_elements[:TARGET_MOVIES]

print('Movies selected for extraction:', len(movie_elements))

Movies selected for extraction: 6000


## 7. Extract Movie Title and Storyline

If a title or storyline is missing for a card, an empty string is stored so that the notebook can continue.

In [8]:
movie_titles = []
storylines = []

for i, movie in enumerate(movie_elements, start=1):
    try:
        title = movie.find_element(
            By.CSS_SELECTOR,
            'h3.ipc-title__text, h4.ipc-title__text'
        ).text.strip()
    except Exception:
        title = ''

    try:
        storyline = movie.find_element(
            By.CSS_SELECTOR,
            'div.ipc-html-content-inner-div'
        ).text.strip()
    except Exception:
        storyline = ''

    movie_titles.append(title)
    storylines.append(storyline)

    if i % 250 == 0:
        print(f'Extracted {i} movies')

print('Extraction completed.')

Extracted 250 movies
Extracted 500 movies
Extracted 750 movies
Extracted 1000 movies
Extracted 1250 movies
Extracted 1500 movies
Extracted 1750 movies
Extracted 2000 movies
Extracted 2250 movies
Extracted 2500 movies
Extracted 2750 movies
Extracted 3000 movies
Extracted 3250 movies
Extracted 3500 movies
Extracted 3750 movies
Extracted 4000 movies
Extracted 4250 movies
Extracted 4500 movies
Extracted 4750 movies
Extracted 5000 movies
Extracted 5250 movies
Extracted 5500 movies
Extracted 5750 movies
Extracted 6000 movies
Extraction completed.


## 8. Create a Pandas DataFrame

In [9]:
df = pd.DataFrame({
    'Movie_Title': movie_titles,
    'Storyline': storylines
})

df.head()

,Movie_Title,Storyline
0,1. Arthur the King,An adventure racer adopts a stray dog named Ar...
1,2. Deadpool & Wolverine,Deadpool is offered a place in the Marvel Cine...
2,3. Gladiator II,After his home is conquered by the tyrannical ...
3,4. Anora,A young stripper from Brooklyn impulsively mar...
4,5. The Ministry of Ungentlemanly Warfare,The British military recruits a small group of...


## 9. Basic Scraping Cleanup

Only basic cleanup is done here. Detailed NLP preprocessing should be done later in the recommendation notebook.

In [10]:
# Remove rows where both title and storyline are empty
df = df[
    (df['Movie_Title'].str.strip() != '') |
    (df['Storyline'].str.strip() != '')
].copy()

# Remove IMDb numbering such as '1. Movie Name'
df['Movie_Title'] = df['Movie_Title'].str.replace(
    r'^\d+\.\s*',
    '',
    regex=True
)

# Remove duplicate movie titles
df = df.drop_duplicates(
    subset='Movie_Title',
    keep='first'
).reset_index(drop=True)

df.head()

,Movie_Title,Storyline
0,Arthur the King,An adventure racer adopts a stray dog named Ar...
1,Deadpool & Wolverine,Deadpool is offered a place in the Marvel Cine...
2,Gladiator II,After his home is conquered by the tyrannical ...
3,Anora,A young stripper from Brooklyn impulsively mar...
4,The Ministry of Ungentlemanly Warfare,The British military recruits a small group of...


## 10. Check the Scraped Dataset

In [11]:
print('Dataset shape:', df.shape)
print('\nMissing values:')
print(df.isnull().sum())

print('\nEmpty storylines:')
print((df['Storyline'].str.strip() == '').sum())

print('\nDuplicate movie titles:')
print(df['Movie_Title'].duplicated().sum())

Dataset shape: (5946, 2)

Missing values:
Movie_Title    0
Storyline      0
dtype: int64

Empty storylines:
215

Duplicate movie titles:
0


## 11. Save the Dataset

In [13]:
# If this notebook is inside the notebooks folder, this saves the CSV inside ../data/
OUTPUT_FILE = '../data/imdb_movies_2024_6000.csv'

df.to_csv(
    OUTPUT_FILE,
    index=False,
    encoding='utf-8-sig'
)

print('CSV saved successfully.')
print('File:', OUTPUT_FILE)

CSV saved successfully.
File: ../data/imdb_movies_2024_6000.csv


## 12. Close Chrome

In [14]:
driver.quit()
print('Browser closed.')

Browser closed.
